### Structured outputs


In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

### Pydantic
pydentic models provide the richest feature set with field validation,discriptions and nested structures

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field( description="The title of the movie")
    year: int = Field( description="The release year of the movie")
    genre: str = Field( description="The genre of the movie")
    director: str = Field( description="The director of the movie")
    rating: float = Field( description="The rating of the movie")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001AC749042F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AC74904D70>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [5]:
model_with_structure.invoke("Tell me about the movie Inside Out")

Movie(title='Inside Out', year=2015, genre='Animation, Family, Adventure', director='Pete Docter', rating=8.1)

In [6]:
model.invoke("Tell me about the movie Inside Out")


AIMessage(content='<think>\nOkay, I need to explain the movie "Inside Out." Let\'s start with the basics. It\'s an animated film by Pixar, released in 2015. Directed by Pete Docter, right? The main characters are the emotions inside a girl\'s mind. The protagonist is Riley, an 11-year-old girl, but the story is about her emotions: Joy, Sadness, Anger, Fear, and Disgust. The main plot is these emotions trying to guide Riley through moving to a new city and dealing with the challenges of growing up.\n\nWait, Joy is the main one, right? She\'s the leader, but she has to work with Sadness. There\'s a conflict when Joy and Sadness get separated from Headquarters, the control center in Riley\'s mind. They travel through different parts of her mind—like Imagination Land, Abstract Thought, and Dream Production. They meet other characters, maybe a guy named Bing Bong who is a memory. I should mention how the movie handles emotions, showing that sadness is important too, not just happiness.\n\nT

In [7]:
response = model_with_structure.invoke("Provide details about the moview Inside Out")
response

Movie(title='Inside Out', year=2015, genre='Animation, Family', director='Pete Docter', rating=8.1)

### Message output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details. """
    title: str = Field( ... , description="The title of the movie")
    year: int = Field( ... , description="The year the movie was released")
    director: str = Field( ... , description="The director of the movie")
    rating: float = Field( ... , description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There's a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception. Let me recall: Inception was directed by Christopher Nolan, released in 2010. The rating is probably around 8.8 on IMDb. I should structure the response with those parameters. Let me verify the details to ensure accuracy. Yeah, that's correct. So I'll use the Movie function with those arguments.\n", 'tool_calls': [{'id': 'wm0rnc7vp', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 232, 'total_tokens': 400, 'completion_time': 0.293575437, 'completion_tokens_details': {'reasoning_tokens': 120}, 'prompt_ti

### Nested Structure


In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")
model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie The Matrix")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie The Matrix. Let me see what I need to do here. The available tool is the MovieDetails function. The function requires title, year, cast, and genres. The user didn't specify the year, but I know The Matrix came out in 1999. I should confirm that. The cast includes Keanu Reeves as Neo, Laurence Fishburne as Morpheus, Carrie-Anne Moss as Trinity, and others. The genres are action, sci-fi, thriller. Budget was around $63 million. Let me structure the response with all these parameters. Make sure to include the required fields first: title, year, cast, genres. Then add the budget if available. Need to format the cast as an array of objects with name and role. Check if all required parameters are present. Alright, I think that's all.\n", 'tool_calls': [{'id': 'g2ffk81qd', 'function': {'arguments': '{"budget":63,"cast":[{"name":"Keanu Reeves","role":"Neo"},{"name":"Laure

### TypedDict


In [10]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details. """
    title: Annotated[str, ... , "The title of the movie"]
    year: Annotated[int, ... , "The year the movie was released"]
    director: Annotated[str, ... , "The director of the movie"]
    rating: Annotated[float, ... , "The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
model_withtypedict.invoke("Please provide the details of the movie 3 idiots")

{'director': 'Rajkumar Hirani',
 'rating': 8.4,
 'title': '3 Idiots',
 'year': 2009}

In [11]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")
model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie The Matrix")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie The Matrix. Let me check the tools available. There\'s a function called MovieDetails that returns information about a movie. The parameters required are title, year, cast, genres, and budget. I need to make sure I include all these details.\n\nFirst, the title is "The Matrix". The year it was released is 1999. For the cast, I should list the main actors with their roles. Keanu Reeves as Neo, Laurence Fishburne as Morpheus, Carrie-Anne Moss as Trinity, and Hugo Weaving as Agent Smith. The genres would be Action, Sci-Fi, and Thriller. The budget was around $63 million.\n\nI need to structure this into the function\'s parameters. Let me verify each parameter type. The cast is an array of dictionaries with name and role. Genres are an array of strings. Budget is a number. All required fields are present. I think that\'s all. Let me call the MovieDetails function with

In [12]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### Dataclasses


In [17]:
import os
from dotenv import load_dotenv
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [19]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
import os
from dotenv import load_dotenv

load_dotenv()

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

# Use Groq instead of gpt-5
model = ChatGroq(model="llama-3.3-70b-versatile")
model_structured = model.with_structured_output(ContactInfo)

# Invoke directly without agent
result = model_structured.invoke(
    "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
)

print(result)

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [21]:
print(result)              # prints the whole object
print(result.name)         # John Doe
print(result.email)        # john@example.com
print(result.phone)        # (555) 123-4567

name='John Doe' email='john@example.com' phone='(555) 123-4567'
John Doe
john@example.com
(555) 123-4567


In [24]:
#dataclass
from dataclasses import dataclass
from langchain_groq import ChatGroq
@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

agent = create_agent(
    model=model,
    response_format=ContactInfo
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}
    ]
})
print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2d79990f-b3b1-4a03-a2b6-ce782b2d05a5'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4z79w8ssz', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 278, 'total_tokens': 311, 'completion_time': 0.056399866, 'completion_tokens_details': None, 'prompt_time': 0.015019359, 'prompt_tokens_details': None, 'queue_time': 0.161158739, 'total_time': 0.071419225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f0281-20b0-7721-a5ba-17a024839e36-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'joh